# Language modeling in NLP?
 - Language model(ing) is the capability of a machine (model) to learn to predict the probability of a sequence of words or characters that are coming next.

## Types of language models in NLP:
 - There are two main types of language models in NLP:
 1. Statistical language models: this type is using statistical techniques such as N-gram language model to predict and generate text.
 2. Nueral language models: this type is based on the deep learning technology to predict and generate text. This type is shown to perform very well in NLP and it works in 2 levels:
      - character level
      - word level

# 1. Building an N-gram Language Model

In [1]:
# 1. lets import libraries
from nltk.corpus import reuters
from nltk import bigrams, trigrams
from collections import Counter, defaultdict
import nltk
nltk.download('reuters')
nltk.download('punkt')

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
# 2. Creating a placeholder for the model
model = defaultdict(lambda: defaultdict(lambda: 0))

In [3]:
# 3. creating a loop to iterate and count the co-occurrence
for sentence in reuters.sents():
  for w1, w2, w3 in trigrams(sentence, pad_right=True, pad_left=True):
    model[(w1, w2)][w3] += 1

In [4]:
# 4. let's transform the counts we created here to probabilities
for w1_w2 in model:
  total_count = float(sum(model[w1_w2].values()))
  for w3 in model[w1_w2]:
    model[w1_w2][w3] /= total_count

In [5]:
#5. Lets display the model output
print(dict(model['today','the']))

{'public': 0.05555555555555555, 'European': 0.05555555555555555, 'Bank': 0.05555555555555555, 'price': 0.1111111111111111, 'emirate': 0.05555555555555555, 'overseas': 0.05555555555555555, 'newspaper': 0.05555555555555555, 'company': 0.16666666666666666, 'Turkish': 0.05555555555555555, 'increase': 0.05555555555555555, 'options': 0.05555555555555555, 'Higher': 0.05555555555555555, 'pound': 0.05555555555555555, 'Italian': 0.05555555555555555, 'time': 0.05555555555555555}


In [6]:
#6. Getting the default dictionary of the model
dict(model['the','price'])

{'yesterday': 0.004651162790697674,
 'of': 0.3209302325581395,
 'it': 0.05581395348837209,
 'effect': 0.004651162790697674,
 'cut': 0.009302325581395349,
 'for': 0.05116279069767442,
 'paid': 0.013953488372093023,
 'to': 0.05581395348837209,
 'increases': 0.013953488372093023,
 'used': 0.004651162790697674,
 'climate': 0.004651162790697674,
 '.': 0.023255813953488372,
 'cuts': 0.009302325581395349,
 'reductions': 0.004651162790697674,
 'limit': 0.004651162790697674,
 'now': 0.004651162790697674,
 'moved': 0.004651162790697674,
 'per': 0.013953488372093023,
 'adjustments': 0.004651162790697674,
 '(': 0.009302325581395349,
 'slumped': 0.004651162790697674,
 'is': 0.018604651162790697,
 'move': 0.004651162790697674,
 'evolution': 0.004651162790697674,
 'differentials': 0.009302325581395349,
 'went': 0.004651162790697674,
 'the': 0.013953488372093023,
 'factor': 0.004651162790697674,
 'Royal': 0.004651162790697674,
 ',': 0.018604651162790697,
 'again': 0.004651162790697674,
 'changes': 0.0

In [7]:
#7. We can display the text generated by the model.

# import library
import random

In [8]:
# let's start with the first words
text = ['today', 'the']
sentence_finished = False

In [9]:
# lets create a condition clause
while not sentence_finished:
  r = random.random()
  accumulator = .0


  for word in model[tuple(text[-2:])].keys():
    accumulator += model[tuple(text[-2:])][word]

    if accumulator >= r:
      text.append(word)
      break

  if text[-2:] == [None, None]:
    sentence_finished = True

In [10]:
print(' '.join([t for t in text if t]))

today the Bank of England said it is expected to continue at relatively high car prices fell 0 . 75 Ecus , will seek a unifying stand from producers , mostly this month , of which was depressed by surplus production .


# 2. Building a Nearual Network

## Importing libraries

In [11]:
# general libraries
import numpy as np
import pandas as pd

# deep learning libraries
from keras.utils import to_categorical
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import LSTM, Dense, GRU, Embedding
from keras. callbacks import EarlyStopping, ModelCheckpoint


#### 2. Loading the Data

In [12]:
data_text = "The unanimous Declaration of the thirteen united States of\
America, When in the Course of human events, it becomes necessary for one\
people to dissolve the political\
bands which have connected them with another, and to assume among the\
powers of the earth, the separate and equal station to which the Laws of\
Nature and of Nature's God entitle them,\
a decent respect to the opinions of mankind requires that they should\
declare the causes which impel them to the separation.\
We hold these truths to be self-evident, that all men are created equal,\
that they are endowed by their Creator with certain unalienable Rights,\
that among these are Life, Liberty and the pursuit of Happiness.--That to\
secure these rights, Governments are instituted among Men, deriving their\
just powers from the consent of the governed,\
--That whenever any Form of Government becomes destructive of these ends,\
it is the Right of the People to alter or to abolish it, and to institute\
new Government,\
laying its foundation on such principles and organizing its powers in such\
form, as to them shall seem most likely to effect their Safety and\
Happiness. Prudence, indeed,\
will dictate that Governments long established should not be changed for\
light and transient causes; and accordingly all experience hath shewn,\
that mankind are more disposed to suffer, while evils are sufferable, than\
to right themselves by abolishing the forms to which they are accustomed.\
But when a long train of abuses and usurpations, pursuing invariably the\
same Object evinces a design to reduce them under absolute Despotism, it\
is their right,\
it is their duty, to throw off such Government, and to provide new Guards\
for their future security.--Such has been the patient sufferance of these\
Colonies;\
and such is now the necessity which constrains them to alter their former\
Systems of Government. The history of the present King of Great Britain is\
a history of\
repeated injuries and usurpations, all having in direct object the\
establishment of an absolute Tyranny over these States. To prove this, let\
Facts be submitted to a candid world.\
General Congress, Assembled, appealing to the Supreme Judge of the world\
for the rectitude of our intentions, do, in the Name, and by Authority of\
the good People of these Colonies,\
solemnly publish and declare, That these United Colonies are, and of Right\
ought to be Free and Independent States;\
that they are Absolved from all Allegiance to the British Crown, and that\
all political connection between them and the State of Great Britain,\
is and ought to be totally dissolved; and that as Free and Independent\
States, they have full Power to levy War, conclude Peace, contract\
Alliances,\
establish Commerce, and to do all other Acts and Things which Independent\
States may of right do. And for the support of this Declaration,\
with a firm reliance on the protection of divine Providence, we mutually\
pledge to each other our Lives, our Fortunes and our sacred Honor."

In [13]:
# First lets create a character mapping
chars = sorted(list(set(data_text))) # Replace 'data_new' with 'data_text'

In [14]:
import re
# The re library in Python stands for "regular expression". It provides a powerful
# set of tools for working with text patterns

In [15]:
# lets define the text_cleaner function

def text_cleaner(text):
  newString = text.lower() # lowercase text
  newString = re.sub(r"'s/b","", newString)
  newString = re.sub("[^a-zA-Z]"," ", newString) # removing any punctation
  long_words = [] # removing short words

  for i in newString.split():
    if len(i) >= 3:
      long_words.append(i)

  return (" ".join(long_words)).strip()


In [16]:
# preprocessing the text
data_new = text_cleaner(data_text)

### Creating Sequences

In [17]:
# kets define a function for creating the sequences

def create_seq(text):
  length = 30 # the length of the sequence
  sequences = list()

  for i in range(length, len(text)):
    seq = text[i-length:i+1] # select sequence of tokens
    sequences.append(seq) # adding more sequence to the list

  print('Total Sequences: %d' % len(sequences))
  return sequences

In [18]:
# creating the sequences
sequences = create_seq(data_text)

Total Sequences: 2895


#### 5. Data(text) preprocessing
1. This is the step, we need to make sure that every data value is converted into numeric data type.
2. This is the step before building the model.
3. In NLP, we call this step as text representation,
4. In DS, we call this step data encoding.

In [19]:
# First lets create a character mapping
chars = sorted(list(set(data_new)))
mapping = dict((c,i) for i, c in enumerate (chars))

# Enumerate in Puthon is used to loop over an iterable and automactially provide
# an index for each item. It returns tuples with the index
# and corresponding value.

In [20]:
# then, we define a function to perform text-encoding
def encode_seq(seq):
  sequences = list()
  for line in seq:
    encoded_seq = [mapping.get(char, -1) for char in line] # encoding every char in line to be an integer, if not found, use -1
    sequences.append(encoded_seq) # adding and storing our sequences into numeric calue
  return sequences


In [21]:
# let's call the function we we just defined: encore_seq
# also, we store the output which are all numeric values now into a variable called sequences
sequences = encode_seq(sequences) # Call the function to encode sequences


In [22]:
L = [1,2,3,4]
L

[1, 2, 3, 4]

In [23]:
type(L)

list

In [24]:
L1 = np.array([1,2,3,4])

In [25]:
type(L1)

numpy.ndarray

#### 6. Model Building
In this step, before building the model, we need to split the data(text) into training and test(validation) sets.


In [26]:
# first, let's import the libraries for model building
from sklearn.model_selection import train_test_split


In [27]:
# then, lets create the vocabulary size
vocab = len(mapping)
sequences = np.array(sequences) # converting numberic values stored in sequences into numpy arrays

In [28]:
# In this step, we split the data into X (training set), y (test set)
X, y = sequences[:,:-1], sequences[:,-1]


# making sure all y values are numeric: one-hot coding
y = to_categorical(y, num_classes=vocab)

# creating train and test (validation) sets
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1)

# display the shape of training and validation sets
print('Train shape:', X_tr.shape, 'Val shape:', X_val.shape)

Train shape: (2605, 30) Val shape: (290, 30)


**In this step, we can build the language model**

In [29]:
# Lets build the lg model
model = Sequential()

model.add(Embedding(vocab, 50, input_length=30, trainable=True))

model.add(GRU(150, recurrent_dropout=0.1, dropout=0.1))

model.add(Dense(vocab, activation='softmax'))

print(model.summary())

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [30]:
# lets compile a categorial
model.compile(loss='categorical_crossentropy', metrics=['acc'], optimizer='adam')

In [31]:
#lets fit the model
model.fit(X_tr, y_tr, epochs=100, verbose=2, validation_data= (X_val, y_val))
# Using verbose while training the model helps to detect
# overfitting which occurs if your acc keeps improving while
# your val_acc gets worse

Epoch 1/100
82/82 - 14s - 170ms/step - acc: 0.1539 - loss: 2.9838 - val_acc: 0.1483 - val_loss: 2.8885
Epoch 2/100
82/82 - 17s - 209ms/step - acc: 0.2031 - loss: 2.7435 - val_acc: 0.2724 - val_loss: 2.6169
Epoch 3/100
82/82 - 8s - 101ms/step - acc: 0.2902 - loss: 2.4719 - val_acc: 0.3103 - val_loss: 2.4259
Epoch 4/100
82/82 - 11s - 130ms/step - acc: 0.3152 - loss: 2.3401 - val_acc: 0.3241 - val_loss: 2.3744
Epoch 5/100
82/82 - 9s - 108ms/step - acc: 0.3248 - loss: 2.2639 - val_acc: 0.3483 - val_loss: 2.3230
Epoch 6/100
82/82 - 8s - 97ms/step - acc: 0.3374 - loss: 2.2021 - val_acc: 0.3310 - val_loss: 2.2890
Epoch 7/100
82/82 - 10s - 126ms/step - acc: 0.3566 - loss: 2.1465 - val_acc: 0.3276 - val_loss: 2.2983
Epoch 8/100
82/82 - 8s - 93ms/step - acc: 0.3754 - loss: 2.0913 - val_acc: 0.3345 - val_loss: 2.3030
Epoch 9/100
82/82 - 5s - 64ms/step - acc: 0.3927 - loss: 2.0283 - val_acc: 0.3414 - val_loss: 2.2858
Epoch 10/100
82/82 - 9s - 116ms/step - acc: 0.4042 - loss: 1.9732 - val_acc: 0.34

### 7. Generating text
- At this step, after the model has been trained, it can genereate text(predicting the next sequence, or word).

In [32]:
# let's define a dunction that generates sequence
def generate_seq(model, mapping, seq_length, seed_text, n_chars):
  in_text = seed_text
  for _ in range(n_chars): # to generate a fixed number of characters
    encoded = [mapping[char]for char in in_text]
  # encode the characters as integers

    encoded = pad_sequences([encoded], maxlen= seq_length, truncating='pre')
  #truncate sequences to a fixed length

  yhat = np.argmax(model.predict(encoded, verbose=0))
  # predict the character

  out_char = ''

  for char, index in mapping.items():
    if index == yhat:
      out_char = char
      break

    in_text = in_text + char # appending the input

  return in_text

In [33]:
inp = 'these armies of the '

In [34]:
len(inp)

20

In [35]:
print(generate_seq(model, mapping, 30, inp.lower(), 15))

these armies of the  abcd


### Conclusion:

1.  Statistical languague models predict word sequences based on frequency and probability
2.  Bigrams are pairs of consecutive words.
3.  N-language models have 2 levels which are the character and word level.
4.  Language modeling allows a machine to learn to predict the probability of a sequence of words.
5.  Using Nltk. bigrams and trigrams breaks the sentence down into sequences of 2 or 3 words.
6.  Defaultdict is used to store word counts.
7.  Model is a 2-level dictionary where the 1st level represents the contect and the second level stores the counts of words that follow.
8.  Nltk.brigrams(sentence) generates all possible bigrams from each sentence.
9.  Total_counts stores the total occurrences of the context word w1.
10. This allows for text generation where the user can start with an initial word input and the model can predict the next word to generae sentences.
11. N-models like transformers are much more equipped to handle more complex language and understood longer parts of a sentence.
12. Data has to be split into training and validation sets before it can be trained by a model.
13. Each data value has to be converted into numeric data types which is called text representation.
14. This is also called data encoding in data science. The data has to be turned into binary code for the model to read it.
15. Each epocj is a complete passthrough of the dataset through the training model and our model had 100 epochs.

